In [1]:
import pandas as pd, numpy as np, blosc2
from pathlib import Path

du = Path("Data/FutureNetwork-data-unlabeled/unlabeled")
dl = Path("Data/FutureNetwork-data-labeled/labeled")
unl = pd.read_csv(du / "samples.csv")
lab = pd.read_csv(dl / "samples.csv")

print("unlabeled rows:", len(unl), "| unique ids:", unl.identifier.nunique())
print(unl.headcount.value_counts())               # should all be 'unlabeled'

# do both files exist for every row?
missing = [i for i in unl.identifier
           if not (du / f"CSI_{i}.b2").exists() or not (du / f"ATI_{i}.b2").exists()]
print("rows missing a file:", len(missing))

print(unl.groupby("layout").agg(rooms=("room", "nunique"), traces=("identifier", "count")))

unlabeled rows: 150 | unique ids: 150
headcount
unlabeled    150
Name: count, dtype: int64
rows missing a file: 0
         rooms  traces
layout                
d            1       8
h            1      14
i            1      17
j            1       4
u            1      55
unknown      1       5
v            1      47


In [2]:
import numpy as np, pandas as pd, blosc2
from pathlib import Path

du = Path("Data/FutureNetwork-data-unlabeled/unlabeled")
unl = pd.read_csv(du / "samples.csv")

W, L, T = [], [], []
for _, r in unl.iterrows():
    csi = blosc2.load_array(str(du / f"CSI_{r.identifier}.b2"))
    for s in range(0, 12000 - 599, 1200):                  # 10 windows per trace
        amp = np.abs(csi[s:s+600, 0]).reshape(600, 3, 64, 16).mean(axis=3)
        W.append(amp.astype(np.float16)); L.append(r.layout); T.append(r.identifier)
np.savez("windows_unl.npz", W=np.array(W), layouts=np.array(L), traces=np.array(T))
print(len(W), "windows")

1500 windows


In [4]:
import numpy as np, torch
from torch import nn
import torch.nn.functional as F

zl = np.load("windows.npz"); zu = np.load("windows_unl.npz")
Wl = torch.tensor(zl["W"].astype(np.float32)); yl = torch.tensor(zl["y"]); lay_l = zl["layouts"]
Wu = torch.tensor(zu["W"].astype(np.float32))

def prep(x):                                    # x: (N, T, 3, 64) raw amplitude
    N, T = x.shape[0], x.shape[1]
    x = x / (x.mean(dim=(1, 2, 3), keepdim=True) + 1e-6)
    x = x - x.mean(dim=1, keepdim=True)
    return x.reshape(N, T, 192).permute(0, 2, 1)

def make_net():
    return nn.Sequential(
        nn.Conv1d(192, 32, 9, stride=2, padding=4), nn.ReLU(),
        nn.Conv1d(32, 32, 9, stride=2, padding=4), nn.ReLU(),
        nn.Conv1d(32, 32, 9, stride=2, padding=4), nn.ReLU(),
        nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(32, 5))

def augment(x):                                 # x: (B, 600, 3, 64)
    B = x.shape[0]
    L = np.random.randint(300, 601); s = np.random.randint(0, 601 - L)
    x = x[:, s:s+L]                                          # random time crop
    x = x * torch.exp(0.2 * torch.randn(B, 1, 3, 1))         # random per-antenna gain
    x = x * (torch.rand(B, 1, 1, 64) > 0.15).float()         # drop subcarrier groups
    return x * (1 + 0.05 * torch.randn_like(x))              # multiplicative noise

def nt_xent(a, b, tau=0.2):
    z = F.normalize(torch.cat([a, b]), dim=1)
    sim = z @ z.T / tau
    sim.fill_diagonal_(-1e9)
    n = a.shape[0]
    target = torch.cat([torch.arange(n, 2*n), torch.arange(0, n)])
    return F.cross_entropy(sim, target)

net = make_net()
encoder = net[:8]                               # conv stack + pooling + flatten -> 32 dims
proj = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 32))
Wpre = torch.cat([Wu, Wl])                      # unlabeled + labeled windows (labels unused)

def pretrain(encoder, proj, data, epochs=20, bs=128):
    params = list(encoder.parameters()) + list(proj.parameters())
    opt = torch.optim.AdamW(params, 1e-3, weight_decay=1e-2)
    for ep in range(epochs):
        encoder.train(); tot = 0
        for idx in torch.randperm(len(data)).split(bs):
            if len(idx) < 8: continue
            x = data[idx]
            loss = nt_xent(proj(encoder(prep(augment(x)))), proj(encoder(prep(augment(x)))))
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        print(f"epoch {ep}: loss {tot / max(1, len(data)//bs):.3f}")

pretrain(encoder, proj, Wpre)
torch.save(net.state_dict(), "cnn_pretrained.pt")

epoch 0: loss 5.752
epoch 1: loss 5.602
epoch 2: loss 5.281
epoch 3: loss 5.196
epoch 4: loss 5.112
epoch 5: loss 4.990
epoch 6: loss 4.951
epoch 7: loss 4.891
epoch 8: loss 4.817
epoch 9: loss 4.845
epoch 10: loss 4.787
epoch 11: loss 4.746
epoch 12: loss 4.716
epoch 13: loss 4.699
epoch 14: loss 4.619
epoch 15: loss 4.582
epoch 16: loss 4.537
epoch 17: loss 4.512
epoch 18: loss 4.420
epoch 19: loss 4.271


In [5]:
import copy
pre_state = torch.load("cnn_pretrained.pt")

def finetune_eval(init_state, held, epochs=30, lr=3e-4):
    te = torch.tensor(lay_l == held); tr = ~te
    Xtr, ytr, Xte, yte = prep(Wl[tr]), yl[tr], prep(Wl[te]), yl[te]
    net = make_net()
    if init_state is not None: net.load_state_dict(init_state)
    opt = torch.optim.AdamW(net.parameters(), lr, weight_decay=1e-2)
    for ep in range(epochs):
        net.train()
        for idx in torch.randperm(len(Xtr)).split(32):
            opt.zero_grad(); F.cross_entropy(net(Xtr[idx]), ytr[idx]).backward(); opt.step()
    net.eval()
    with torch.no_grad(): pred = net(Xte).argmax(1)
    return (pred == yte).float().mean().item(), torch.bincount(pred, minlength=5).tolist()

for held in ["a", "c", "g"]:
    s = finetune_eval(None, held); p = finetune_eval(pre_state, held)
    print(f"held {held}: scratch {s[0]:.2f} {s[1]} | pretrained {p[0]:.2f} {p[1]}")

held a: scratch 0.05 [66, 96, 2, 0, 76] | pretrained 0.04 [35, 120, 1, 1, 83]
held c: scratch 0.11 [91, 136, 13, 0, 0] | pretrained 0.17 [28, 148, 64, 0, 0]
held g: scratch 0.33 [0, 143, 0, 2, 195] | pretrained 0.22 [1, 76, 15, 7, 241]


In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
encoder.eval()
with torch.no_grad(): E = encoder(prep(Wl)).numpy()
print("predict layout   from embedding:", cross_val_score(KNeighborsClassifier(5), E, lay_l, cv=3).mean())
print("predict headcount from embedding:", cross_val_score(KNeighborsClassifier(5), E, yl.numpy(), cv=3).mean())

predict layout   from embedding: 0.5343749999999999
predict headcount from embedding: 0.340625


In [7]:
net = make_net(); net.load_state_dict(pre_state)
X = prep(Wl)
opt = torch.optim.AdamW(net.parameters(), 3e-4, weight_decay=1e-2)
for ep in range(30):
    net.train()
    for idx in torch.randperm(len(X)).split(32):
        opt.zero_grad(); F.cross_entropy(net(X[idx]), yl[idx]).backward(); opt.step()
torch.save(net.state_dict(), "cnn_ssl.pt")

In [8]:
import torch, numpy as np, onnxruntime as ort
from torch import nn
from time import perf_counter

class Submission(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net
    def forward(self, frames, intervals):
        f = frames[:, 0]                                        # (600, 3, 1024, 2)
        amp = torch.sqrt(f[..., 0] ** 2 + f[..., 1] ** 2)       # (600, 3, 1024)
        amp = amp.reshape(600, 3, 64, 16).mean(dim=3)           # (600, 3, 64)
        amp = amp / (amp.mean() + 1e-6)
        amp = amp - amp.mean(dim=0, keepdim=True)
        x = amp.reshape(600, 192).permute(1, 0).unsqueeze(0)    # (1, 192, 600)
        return self.net(x)[0]                                   # (5,)

net.load_state_dict(torch.load("cnn_ssl.pt")); net.eval()
model = Submission(net).eval()

frames = torch.randn(600, 1, 3, 1024, 2)
intervals = torch.full((600,), 5000.0)
torch.onnx.export(model, (frames, intervals), "model.onnx",
                  input_names=["frames", "intervals"], output_names=["logits"],
                  opset_version=18, external_data=False)

W1020 23:07:24.696000 26608 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::nms
W1020 23:07:24.698000 26608 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::roi_align
W1020 23:07:24.699000 26608 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::roi_pool
W1020 23:07:24.700000 26608 Lib\site-packages\torch\onnx\_internal\exporter\_registration.py:107] torchvision is not installed. Skipping torchvision::deform_conv2d


[torch.onnx] Obtain model graph for `Submission([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Submission([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.14.1+cpu',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"frames"<FLOAT,[600,1,3,1024,2]>,
                %"intervals"<FLOAT,[600]>
            ),
            outputs=(
                %"logits"<FLOAT,[5]>
            ),
            initializers=(
                %"net.0.weight"<FLOAT,[32,192,9]>{TorchTensor(...)},
                %"net.0.bias"<FLOAT,[32]>{TorchTensor(...)},
                %"net.2.weight"<FLOAT,[32,32,9]>{TorchTensor(...)},
                %"net.2.bias"<FLOAT,[32]>{TorchTensor(...)},
                %"net.4.weight"<FLOAT,[32,32,9]>{TorchTensor(...)},
                %"net.4.bias"<FLOAT,[32]>{TorchTensor(...)},
                %"net.9.weight"<FLOAT,[5,32]>{TorchTensor(...)},
                %"net.9.bias"

In [ ]:
import blosc2, numpy as np, pandas as pd, torch
from pathlib import Path
from time import perf_counter

d = Path("Data/FutureNetwork-data-labeled/unlabeled")
i = lab.identifier.iloc[0]
csi = blosc2.load_array(str(d / f"CSI_{i}.b2"))
ati = blosc2.load_array(str(d / f"ATI_{i}.b2"))

real_frames = np.ascontiguousarray(csi[:600]).view(np.float32).reshape(600, 1, 3, 1024, 2)
real_intervals = ati[:600].copy()
real_intervals[-1] = 0

print([x.name for x in sess.get_inputs()])   # should list frames AND intervals
out = sess.run(["logits"], {"frames": real_frames, "intervals": real_intervals})[0]
ref = model(torch.from_numpy(real_frames), torch.from_numpy(real_intervals)).detach().numpy()
print(out, ref)

t = perf_counter()
sess.run(["logits"], {"frames": real_frames, "intervals": real_intervals})
print("time:", perf_counter() - t)

FileNotFoundError: [Errno 2] No such file or directory: 'Data\\FutureNetwork-data-labeled\\unlabeled\\samples.csv'